# CDS Instrument Tear Sheet

This notebook renders a single-page HTML tear sheet for a **Credit Default Swap (CDS)**
using the polymorphic `reporting.instrument_tearsheet` API — pass an instrument JSON plus
a `MarketContext` and `as_of` date, and the library prices it with the recommended metric
set and renders the full report inline.

**Market setup:**
- A USD-OIS discount curve (5 knots, 2025-01-02 base).
- A corporate hazard-rate curve (`CORP-HAZARD`, 40 % recovery) — the credit/survival market.

**Pricing model:** `hazard_rate` — the ISDA-standard intensity model for CDS.

The rendered sheet includes credit-specific sections:
- **Key Metrics:** Par Spread, CS01, Risky PV01, Protection/Premium Leg PVs.
- **Bucketed CS01:** per-tenor credit sensitivity profile.
- **Payoff / Survival:** jump-to-default, expected loss, default01, recovery01.

Tooltips are available natively in Jupyter. Open a saved `.html` file in a browser for
the richer crosshair + cursor-following tooltip.

In [ ]:
import datetime as dt
import json
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, HazardCurve, MarketContext
from finstack_quant.valuations.instruments.credit_derivatives import CreditDefaultSwap
from finstack_quant import reporting

In [ ]:
# USD-OIS discount curve anchored to 2025-01-02.
ois = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 2),
    [(0.0, 1.0), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)

# Corporate hazard-rate curve: flat-ish term structure around 2–3 % annual intensity.
hazard = HazardCurve(
    "CORP-HAZARD",
    date(2025, 1, 2),
    [(0.5, 0.018), (1.0, 0.020), (3.0, 0.024), (5.0, 0.028), (10.0, 0.032)],
    recovery_rate=0.40,
)

mc = MarketContext().insert(ois).insert(hazard)

# CreditDefaultSwap.example() returns a ready 5-year USD CDS (pay protection, 100 bp running).
cds = json.loads(CreditDefaultSwap.example().to_json())
print(f"Instrument type : {cds['type']}")
print(f"Reference       : {cds['spec']['id']}")
print(f"Maturity        : {cds['spec']['premium']['end']}")
print(f"Running spread  : {cds['spec']['premium']['spread_bp']} bp")
print(f"Recovery rate   : {cds['spec']['protection']['recovery_rate'] * 100:.0f}%")

In [ ]:
# Render the full tear sheet in one call.
# instrument_tearsheet detects the instrument JSON, prices with recommended CDS metrics
# (par_spread, cs01, bucketed_cs01, jump_to_default, …) using model='hazard_rate',
# and renders the result inline via _repr_html_.
reporting.instrument_tearsheet(
    cds,
    market=mc,
    as_of="2025-01-02",
    model="hazard_rate",
    generated=dt.date(2026, 6, 19),
)

To save the tear sheet as a standalone HTML file:

```python
ts = reporting.instrument_tearsheet(cds, market=mc, as_of="2025-01-02", model="hazard_rate")
ts.save("cds_tearsheet.html")
```